In [5]:
import duckdb
import pandas as pd

# connect database
con = duckdb.connect("fno.db")

# load csv
df = pd.read_csv(r"C:\Users\Sumit\OneDrive\Documents\Desktop\data analyst ass\3mfanddo.csv")

# create table
con.execute("""
CREATE TABLE trades AS
SELECT * FROM df
""")

print(con.execute("SELECT COUNT(*) FROM trades").fetchall())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[(2533210,)]


In [7]:
print(con.execute("DESCRIBE trades").fetchall())

[('Unnamed: 0', 'BIGINT', 'YES', None, None, None), ('INSTRUMENT', 'VARCHAR', 'YES', None, None, None), ('SYMBOL', 'VARCHAR', 'YES', None, None, None), ('EXPIRY_DT', 'VARCHAR', 'YES', None, None, None), ('STRIKE_PR', 'DOUBLE', 'YES', None, None, None), ('OPTION_TYP', 'VARCHAR', 'YES', None, None, None), ('OPEN', 'DOUBLE', 'YES', None, None, None), ('HIGH', 'DOUBLE', 'YES', None, None, None), ('LOW', 'DOUBLE', 'YES', None, None, None), ('CLOSE', 'DOUBLE', 'YES', None, None, None), ('SETTLE_PR', 'DOUBLE', 'YES', None, None, None), ('CONTRACTS', 'DOUBLE', 'YES', None, None, None), ('VAL_INLAKH', 'DOUBLE', 'YES', None, None, None), ('OPEN_INT', 'DOUBLE', 'YES', None, None, None), ('CHG_IN_OI', 'DOUBLE', 'YES', None, None, None), ('TIMESTAMP', 'VARCHAR', 'YES', None, None, None)]


In [8]:
df.columns


Index(['Unnamed: 0', 'INSTRUMENT', 'SYMBOL', 'EXPIRY_DT', 'STRIKE_PR',
       'OPTION_TYP', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'SETTLE_PR', 'CONTRACTS',
       'VAL_INLAKH', 'OPEN_INT', 'CHG_IN_OI', 'TIMESTAMP'],
      dtype='object')

In [9]:
con.execute("""
ALTER TABLE trades
ADD COLUMN exchange VARCHAR
""")

In [10]:
con.execute("""
UPDATE trades
SET exchange='NSE'
""")

In [11]:
con.execute("""
SELECT exchange,
AVG(settle_pr) as avg_settle_price
FROM trades
GROUP BY exchange
""").df()

,exchange,avg_settle_price
0,NSE,321.040575


In [13]:
con.execute("DESCRIBE trades").df()

,column_name,column_type,null,key,default,extra
0,Unnamed: 0,BIGINT,YES,None,None,None
1,INSTRUMENT,VARCHAR,YES,None,None,None
2,SYMBOL,VARCHAR,YES,None,None,None
3,EXPIRY_DT,VARCHAR,YES,None,None,None
4,STRIKE_PR,DOUBLE,YES,None,None,None
5,OPTION_TYP,VARCHAR,YES,None,None,None
6,OPEN,DOUBLE,YES,None,None,None
7,HIGH,DOUBLE,YES,None,None,None
8,LOW,DOUBLE,YES,None,None,None
9,CLOSE,DOUBLE,YES,None,None,None


In [14]:
df.columns

Index(['Unnamed: 0', 'INSTRUMENT', 'SYMBOL', 'EXPIRY_DT', 'STRIKE_PR',
       'OPTION_TYP', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'SETTLE_PR', 'CONTRACTS',
       'VAL_INLAKH', 'OPEN_INT', 'CHG_IN_OI', 'TIMESTAMP'],
      dtype='object')

In [15]:
con.execute("""
SELECT expiry_dt,
strike_pr,
SUM(contracts) as total_volume
FROM trades
GROUP BY expiry_dt, strike_pr
ORDER BY expiry_dt
""").df()

,EXPIRY_DT,STRIKE_PR,total_volume
0,01-Aug-2019,31500.0,158.0
1,01-Aug-2019,28900.0,1573405.0
2,01-Aug-2019,28300.0,3518340.0
3,01-Aug-2019,29000.0,1697885.0
4,01-Aug-2019,12000.0,2945.0
...,...,...,...
9114,31-Oct-2019,26100.0,267.0
9115,31-Oct-2019,110.0,85850.0
9116,31-Oct-2019,140.0,139111.0
9117,31-Oct-2019,170.0,87619.0


In [19]:
df = df.rename(columns={"contracts":"volume"})

In [22]:
con.execute("""
SELECT symbol,
MAX(contracts) as max_volume
FROM trades
WHERE STRPTIME(timestamp,'%d-%b-%Y') >= 
(
SELECT MAX(STRPTIME(timestamp,'%d-%b-%Y')) FROM trades
) - INTERVAL 30 DAY
GROUP BY symbol
ORDER BY max_volume DESC
""").df()

,SYMBOL,max_volume
0,BANKNIFTY,4564524.0
1,NIFTY,1274605.0
2,YESBANK,215823.0
3,INFY,84779.0
4,SBIN,83986.0
...,...,...
147,HEXAWARE,3101.0
148,GMRINFRA,2935.0
149,NIITTECH,2565.0
150,OIL,1543.0


In [24]:
df = df.rename(columns={"contracts":"volume"})

In [25]:
con.execute("""
SELECT symbol,
SUM(contracts) as total_volume
FROM trades
GROUP BY symbol
ORDER BY total_volume DESC
LIMIT 10
""").df()

,SYMBOL,total_volume
0,BANKNIFTY,1.003241e+09
1,NIFTY,4.406550e+08
2,YESBANK,1.016104e+07
3,RELIANCE,7.781761e+06
4,SBIN,6.184575e+06
5,MARUTI,5.317907e+06
6,IBULHSGFIN,5.123768e+06
7,ICICIBANK,4.672577e+06
8,HDFCBANK,4.335332e+06
9,TATASTEEL,3.393399e+06


In [27]:
con.execute("""
EXPLAIN
SELECT symbol,
MAX(contracts)
FROM trades
GROUP BY symbol
""").fetchall()

[('physical_plan',
  '┌───────────────────────────┐\n│         PROJECTION        │\n│    ────────────────────   │\n│__internal_decompress_strin│\n│           g(#0)           │\n│             #1            │\n│                           │\n│         ~176 rows         │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│       HASH_GROUP_BY       │\n│    ────────────────────   │\n│         Groups: #0        │\n│    Aggregates: max(#1)    │\n│                           │\n│         ~176 rows         │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│         PROJECTION        │\n│    ────────────────────   │\n│           SYMBOL          │\n│         CONTRACTS         │\n│                           │\n│      ~2,533,210 rows      │\n└─────────────┬─────────────┘\n┌─────────────┴─────────────┐\n│         PROJECTION        │\n│    ────────────────────   │\n│__internal_compress_string_│\n│        uhugeint(#0)       │\n│             #1            │\n│                